# Section 1: Introduction

Welcome to the **HeatSync Preprocessing Walkthrough**.

This notebook demonstrates the preprocessing pipeline that transforms raw telemetry, interval data, power draw recordings, and space schedule events into clean, aligned, 15-minute resolution datasets.

### Objectives
1. **Load Raw Data:** Load the heating and cooling window datasets and perform initial inspection.
2. **Data Quality Overview:** Assess missing values, check for duplicate timestamps, and profile row counts/date ranges.
3. **15-Minute Alignment:** Align all datasets to a common 15-minute grid using the resampling/interpolation rules defined in `utils.py`.
4. **Feature Engineering:** Compute occupancy flags from schedule events, time features (hour of day, day of week, weekend indicator), and target temperature delta indicators.
5. **Alignment Sanity Plots:** Generate plots to visually inspect the alignment quality.
6. **Process & Cache both windows:** Output clean CSV files to the `backend/data/processed/` directory for downstream analysis.

# Section 2: Load Raw Data

In this section, we load the raw data for the heating window using the helper functions in `backend/notebooks/utils.py`.

In [ ]:
import sys
import os
# Adjust paths so we can import our notebooks.utils package
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('backend'))

from notebooks.utils import (
    load_snapshots,
    load_intervals,
    load_power_draw,
    load_space_events,
    build_aligned_dataset,
    save_processed,
    PROJECT_ROOT,
)
import pandas as pd

# Load raw datasets for the heating window
snapshots_raw = load_snapshots('heating')
intervals_raw = load_intervals('heating')
power_raw = load_power_draw('heating')
events_raw = load_space_events('heating')

### Inspect Raw Data Shapes and Types

We display `.head()`, `.shape`, `.dtypes`, and date ranges for each raw DataFrame to understand the loaded structure.

In [ ]:
def display_df_profile(name, df, is_indexed_by_time=True, time_col=None):
    print(f"=== Profiling: {name} ===")
    print("Shape:", df.shape)
    if is_indexed_by_time:
        print("Date Range:", df.index.min(), "to", df.index.max())
    elif time_col:
        print("Date Range:", df[time_col].min(), "to", df[time_col].max())
    else:
        # Try event starts/ends
        print("Date Range:", df['starts_at'].min(), "to", df['ends_at'].max())
    print("\nData Types:")
    print(df.dtypes)
    print("\nFirst 2 Rows:")
    display(df.head(2))
    print("=" * 40 + "\n")

display_df_profile("Snapshots", snapshots_raw, is_indexed_by_time=True)
display_df_profile("Intervals", intervals_raw, is_indexed_by_time=True)
display_df_profile("Power Draw", power_raw, is_indexed_by_time=False, time_col='timestamp')
display_df_profile("Space Events", events_raw, is_indexed_by_time=False)

# Section 3: Data Quality Overview

We inspect the data quality before starting alignment. Specifically, we check missing value counts, duplicate timestamps, raw row counts, and date ranges.

In [ ]:
print("=== Missing Value Counts per Column ===")
for name, df in [('Snapshots', snapshots_raw), ('Intervals', intervals_raw), ('Power Draw', power_raw), ('Space Events', events_raw)]:
    print(f"\n{name}:")
    missing = df.isna().sum()
    print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

print("\n=== Duplicate Timestamp Check (per device) ===")
snap_dups = snapshots_raw.reset_index().duplicated(subset=['device_id', 'last_seen_at']).sum()
int_dups = intervals_raw.reset_index().duplicated(subset=['device_id', 'interval_start_time']).sum()
pwr_dups = power_raw.duplicated(subset=['device_id', 'timestamp']).sum()
print(f"Snapshots duplicate keys: {snap_dups}")
print(f"Intervals duplicate keys: {int_dups}")
print(f"Power Draw duplicate keys: {pwr_dups}")

# Section 4: 15-Minute Alignment

We resample and align the telemetry to a 15-minute grid. If gaps are 2 intervals (30 minutes) or less, we interpolate (forward fill) values. If gaps are longer, we leave them blank and mark them with a `_gap_flag` column.

Let's execute `build_aligned_dataset('heating')` and compare the row counts and read the quality report.

In [ ]:
aligned_heating, quality_report = build_aligned_dataset('heating')
print("Aligned dataset shape:", aligned_heating.shape)
print(f"Before resampling (snapshots count): {len(snapshots_raw)}")
print(f"After resampling (aligned count): {len(aligned_heating)}")

print("\nQuality Report:")
import pprint
pprint.pprint(quality_report)

# Section 5: Feature Engineering Review

Feature engineering adds time features (`hour_of_day`, `day_of_week`, `is_weekend`), occupancy flags derived from the overlapping space events, outside temperature, and target/current temperature delta. Let's inspect these columns.

In [ ]:
engineered_cols = [
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'occupancy_flag',
    'outside_temperature',
    'temperature_delta'
]
# Keep only columns that are present
existing_cols = [c for c in engineered_cols if c in aligned_heating.columns]

print("Sample rows for engineered columns:")
display(aligned_heating[existing_cols].head())

print("\nValue distributions:")
for col in existing_cols:
    print(f"\n--- {col} ---")
    print(aligned_heating[col].describe())

# Section 6: Alignment Sanity Plots

We visualize a subset of the aligned data for a single device to check that our resampling and feature engineering behave correctly.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Get the first device ID
device_id = aligned_heating['device_id'].dropna().unique()[0]
dev_raw_snap = snapshots_raw[snapshots_raw['device_id'] == device_id]
dev_aligned = aligned_heating[aligned_heating['device_id'] == device_id]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Subplot 1: Temperature comparison
axes[0].plot(dev_raw_snap.index, dev_raw_snap['status_temperature_outside_in_celsius'], 'o', color='blue', label='Raw Temp Outside', alpha=0.6)
axes[0].plot(dev_aligned.index, dev_aligned['outside_temperature'], '-', color='cyan', label='Aligned Temp Outside', alpha=0.8)
axes[0].set_title(f"Outside Temperature: Raw vs Aligned (Device: {device_id})")
axes[0].legend()
axes[0].grid(True)

# Subplot 2: Power Draw
axes[1].plot(dev_aligned.index, dev_aligned['power_draw_kw'], '-', color='orange', label='Power Draw (kW)')
axes[1].set_title(f"Power Draw (Device: {device_id})")
axes[1].legend()
axes[1].grid(True)

# Subplot 3: Occupancy Flag
axes[2].fill_between(dev_aligned.index, dev_aligned['occupancy_flag'], color='green', alpha=0.3, label='Occupancy Active')
axes[2].set_title("Occupancy Schedule Flag")
axes[2].legend()
axes[2].grid(True)

plt.xlabel("Timestamp")
plt.tight_layout()
plt.show()

# Section 7: Process Both Windows

We build and save the aligned datasets for both the heating and cooling windows. We save each using the shared utility helper `save_processed()` and output to the expected paths.

In [ ]:
# Run for heating
heating_final, heating_report = build_aligned_dataset('heating')
save_processed(heating_final, 'heating')
heating_final.reset_index().to_csv(PROJECT_ROOT / 'backend/data/processed/processed_heating.csv', index=False)

# Run for cooling
cooling_final, cooling_report = build_aligned_dataset('cooling')
save_processed(cooling_final, 'cooling')
cooling_final.reset_index().to_csv(PROJECT_ROOT / 'backend/data/processed/processed_cooling.csv', index=False)

print("Heating final dataset shape:", heating_final.shape)
print("Cooling final dataset shape:", cooling_final.shape)
print("\nProcessed column list:")
print(list(heating_final.columns))

# Section 8: Summary

We print the coverage statistics for the processed heating and cooling datasets.

In [ ]:
def print_summary_stats(df, name):
    metadata = df.attrs.get('interpolation_metadata', {})
    total_interpolations = sum(info.get('interpolated_count', 0) for info in metadata.values())
    print(f"=== {name} Dataset Summary ===")
    print(f"Total Row Count: {len(df)}")
    print(f"Date Range:      {df.index.min()} to {df.index.max()}")
    print(f"Column Count:    {len(df.columns)}")
    print(f"Total Interpolated Values: {total_interpolations}")
    print()

print_summary_stats(heating_final, "Heating")
print_summary_stats(cooling_final, "Cooling")